In [1]:
import os, sys, requests
from pathlib import Path
# from urllib.parse import urlparse
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
# Cria a conexão Spark

# 1. Remove qualquer barreira de proxy local que jogue o tráfego para a rede da empresa
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# 2. Garante que o Spark use o Python correto do venv
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Força o IP local estrito
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 4. Inicializa configurando a autenticação local do Worker
spark = SparkSession.builder \
    .appName("TesteLocal") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.auth.enabled", "false") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_tempetatura = spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_temperatura.parquet")
df_tempetatura.printSchema()
df_tempetatura.limit(10).show()

root
 |-- data_medicao: date (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+------------+--------+---------+-----------+------------------+--------------+
|data_medicao|latitude|longitude|  indicador|             valor|unidade_medida|
+------------+--------+---------+-----------+------------------+--------------+
|  2025-01-01|     6.0|    -74.0|temperatura|21.317346191406273|       celsius|
|  2025-01-01|     6.0|   -73.75|temperatura|17.715447998046898|       celsius|
|  2025-01-01|     6.0|    -73.5|temperatura|16.692193603515648|       celsius|
|  2025-01-01|     6.0|   -73.25|temperatura|15.932000732421898|       celsius|
|  2025-01-01|     6.0|    -73.0|temperatura|13.354058837890648|       celsius|
|  2025-01-01|     6.0|   -72.75|temperatura|11.741571044921898|       celsius|
|  2025-01-01|     6.0|    

Obtem as estatísticas de temperatura:
- Temperatura mínima
- Temperatura máxima
- Temperatura média
- Percentil 5%
- Percentil 90%

Os dados estão agrupados por ano e mês

In [4]:
df_base = (
    df_tempetatura
        .withColumn("ano", F.year("data_medicao"))
        .withColumn("mes", F.month("data_medicao"))
)

df_stats = (
    df_base
    .groupBy(
        "ano",
        "mes",
        "latitude",
        "longitude"
    )
    .agg(
        F.min("valor").alias("temp_min_mes"),
        F.max("valor").alias("temp_max_mes"),
        F.avg("valor").alias("temp_media_mes"),
        F.expr("percentile_approx(valor, 0.05)").alias("percentil_05_mes"),
        F.expr("percentile_approx(valor, 0.90)").alias("percentil_90_mes")
    )
)

df_stats.printSchema()
df_stats.show(10, truncate=False)

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)
 |-- percentil_05_mes: double (nullable = true)
 |-- percentil_90_mes: double (nullable = true)

+----+---+--------+---------+------------------+------------------+------------------+------------------+------------------+
|ano |mes|latitude|longitude|temp_min_mes      |temp_max_mes      |temp_media_mes    |percentil_05_mes  |percentil_90_mes  |
+----+---+--------+---------+------------------+------------------+------------------+------------------+------------------+
|2025|1  |-34.0   |-73.75   |14.811090087890648|18.047540283203148|16.617592891570084|15.427606201171898|17.296075439453148|
|2025|1  |-34.0   |-72.75   |15.337213134765648|17.719506835937523|16.555147035660305|15.579064941406273|17.040

In [ ]:
# # Salva o Dataframe como csv
# (df_stats
#     .toPandas()
#     .to_csv(r"C:\Marco Conti\Projetos\MAIS-v2\dados\tb_temperatura_mensal.csv"
#            ,index=False
#            ,sep=";"))

In [5]:
# Anexar dados estatíscos e percentis ao dado diário
# Será utilizados para calcular as ondas de calor (periodos de dias consecutivos)
df_dia = (
    df_base.join(
        df_stats.select(
            "ano",
            "mes",
            "latitude",
            "longitude",
            "temp_min_mes",
            "temp_max_mes",
            "temp_media_mes",
            "percentil_05_mes",
            "percentil_90_mes"
        ),
        [
            "ano",
            "mes",
            "latitude",
            "longitude"
        ],
        how="left"
    )
)

print("Número de registros no DataFrame diário:", df_dia.count()) # 151662
df_dia.printSchema()
df_dia.show(10, truncate=False)

Número de registros no DataFrame diário: 783587
root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)
 |-- percentil_05_mes: double (nullable = true)
 |-- percentil_90_mes: double (nullable = true)

+----+---+--------+---------+------------+-----------+------------------+--------------+------------------+------------------+------------------+------------------+------------------+
|ano |mes|latitude|longitude|data_medicao|indicador  |valor             |unidade_medida|temp_min_mes      |temp_max_mes      |temp_media_mes    |percentil_05_mes  |percentil_90_mes  |
+----+---+--------+---------+

In [6]:
# Classificar extremos
df_dia = (
    df_dia
    .withColumn(
        "extremo_alto",
        F.when(F.col("valor") > F.col("percentil_90_mes")
              ,(F.col("valor") - F.col("percentil_90_mes"))).otherwise(0))
    .withColumn(
        "extremo_baixo",
        F.when(F.col("valor") < F.col("percentil_05_mes")
              ,(F.col("percentil_05_mes") - F.col("valor"))).otherwise(0)
    )
)

df_dia.printSchema()
df_dia.filter("extremo_baixo != 0 or extremo_alto != 0").show(10, truncate=False)
# (df_dia
#     .select('data_medicao', 'latitude', 'longitude', 'percentil_05_mes', 'valor', 'percentil_90_mes'
#            ,'extremo_alto', 'extremo_baixo' )
#     .orderBy("latitude", "longitude", "data_medicao")
#     .show(1000, truncate=False))

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)
 |-- percentil_05_mes: double (nullable = true)
 |-- percentil_90_mes: double (nullable = true)
 |-- extremo_alto: double (nullable = true)
 |-- extremo_baixo: double (nullable = true)

+----+---+--------+---------+------------+-----------+------------------+--------------+------------------+------------------+------------------+------------------+------------------+-----------------+-------------+
|ano |mes|latitude|longitude|data_medicao|indicador  |valor             |unidade_medida|temp_min_mes      |temp_max_mes      |temp_media_mes

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------------------------------------------------------
# ETAPA 1: Flag de dia quente e filtro dos registros
# -------------------------------------------------------------------------
# Considera-se dia quente quando a temperatura excede o percentil 90 do mês
df_flag = df_dia.withColumn(
    "flag_dia_quente",
    F.when(F.col("valor") > F.col("percentil_90_mes"), 1).otherwise(0)
)

# Mantém apenas os dias que atenderam ao critério
df_quentes = df_flag.filter(F.col("flag_dia_quente") == 1)


# -------------------------------------------------------------------------
# ETAPA 2: Técnica Gaps & Islands (Agrupamento de dias consecutivos)
# -------------------------------------------------------------------------
# Janela ordenada por data para cada ponto geográfico
janela_loc = Window.partitionBy("latitude", "longitude").orderBy("data_medicao")

# Ao subtrair a ordem do registro (rn) da data, dias consecutivos geram
# exatamente o mesmo identificador de grupo (grupo_id)
df_eventos = df_quentes \
    .withColumn("rn", F.row_number().over(janela_loc)) \
    .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))"))


+--------+---------+--------+------------+-----------+--------+---+---+
|latitude|longitude|grupo_id|duracao_dias|data_inicio|data_fim|ano|mes|
+--------+---------+--------+------------+-----------+--------+---+---+
+--------+---------+--------+------------+-----------+--------+---+---+



In [10]:
# -------------------------------------------------------------------------
# ETAPA 3: Consolidação dos Eventos e Regra do Limiar (> 4 dias)
# -------------------------------------------------------------------------
df_ondas = df_eventos.groupBy("latitude", "longitude", "grupo_id") \
    .agg(
        F.count("data_medicao").alias("duracao_dias"),
        F.min("data_medicao").alias("data_inicio"),
        F.max("data_medicao").alias("data_fim"),
        F.min("ano").alias("ano"),
        F.min("mes").alias("mes")
    ) \
    .filter(F.col("duracao_dias") > 2)  # Onda de calor requer > 4 dias consecutivos


df_ondas.show(10, truncate=False)

+--------+---------+----------+------------+-----------+----------+----+---+
|latitude|longitude|grupo_id  |duracao_dias|data_inicio|data_fim  |ano |mes|
+--------+---------+----------+------------+-----------+----------+----+---+
|-32.5   |-63.75   |2025-01-13|3           |2025-01-14 |2025-01-16|2025|1  |
|-32.5   |-46.75   |2025-01-22|3           |2025-01-23 |2025-01-25|2025|1  |
|-25.75  |-45.0    |2025-01-19|3           |2025-01-20 |2025-01-22|2025|1  |
|-25.5   |-43.25   |2025-01-19|3           |2025-01-20 |2025-01-22|2025|1  |
|-20.75  |-41.25   |2025-01-19|3           |2025-01-20 |2025-01-22|2025|1  |
|-20.5   |-53.0    |2025-01-07|3           |2025-01-08 |2025-01-10|2025|1  |
|-18.75  |-43.5    |2025-01-19|3           |2025-01-20 |2025-01-22|2025|1  |
|-18.5   |-67.25   |2024-12-31|3           |2025-01-01 |2025-01-03|2025|1  |
|-17.75  |-68.5    |2024-12-31|3           |2025-01-01 |2025-01-03|2025|1  |
|-17.5   |-48.5    |2025-01-23|3           |2025-01-24 |2025-01-26|2025|1  |

In [11]:
# Agregação Mensal
df_metricas_mensal = df_ondas.groupBy("latitude", "longitude", "ano", "mes") \
    .agg(
        # Número de eventos no mês
        F.count("grupo_id").alias("numero_ondas_calor"),
        
        # Frequência: Total de dias acumulados em onda de calor no mês
        F.sum("duracao_dias").alias("frequencia_dias_onda_calor"),
        
        # Duração: Média e Máxima das ondas ocorridas no mês
        F.round(F.avg("duracao_dias"), 2).alias("duracao_media_dias"),
        F.max("duracao_dias").alias("duracao_maxima_dias")
    ) \
    .orderBy("ano", "mes", "latitude", "longitude")

df_metricas_mensal.show()

+--------+---------+----+---+------------------+--------------------------+------------------+-------------------+
|latitude|longitude| ano|mes|numero_ondas_calor|frequencia_dias_onda_calor|duracao_media_dias|duracao_maxima_dias|
+--------+---------+----+---+------------------+--------------------------+------------------+-------------------+
|   -34.0|    -72.0|2025|  1|                 1|                         3|               3.0|                  3|
|   -34.0|    -70.0|2025|  1|                 1|                         3|               3.0|                  3|
|   -34.0|   -69.75|2025|  1|                 1|                         3|               3.0|                  3|
|   -34.0|    -66.0|2025|  1|                 1|                         3|               3.0|                  3|
|   -34.0|   -65.75|2025|  1|                 1|                         3|               3.0|                  3|
|   -34.0|    -65.5|2025|  1|                 1|                         3|     

In [12]:
# Agregação Anual
df_metricas_anual = df_ondas.groupBy("latitude", "longitude", "ano") \
    .agg(
        # Número de eventos no ano
        F.count("grupo_id").alias("numero_ondas_calor"),
        
        # Frequência: Total de dias do ano passados em onda de calor
        F.sum("duracao_dias").alias("frequencia_dias_onda_calor"),
        
        # Duração: Média e Máxima das ondas ocorridas no ano
        F.round(F.avg("duracao_dias"), 2).alias("duracao_media_dias"),
        F.max("duracao_dias").alias("duracao_maxima_dias")
    ) \
    .orderBy("ano", "latitude", "longitude")

df_metricas_anual.show()

+--------+---------+----+------------------+--------------------------+------------------+-------------------+
|latitude|longitude| ano|numero_ondas_calor|frequencia_dias_onda_calor|duracao_media_dias|duracao_maxima_dias|
+--------+---------+----+------------------+--------------------------+------------------+-------------------+
|   -34.0|    -72.0|2025|                 1|                         3|               3.0|                  3|
|   -34.0|    -70.0|2025|                 1|                         3|               3.0|                  3|
|   -34.0|   -69.75|2025|                 1|                         3|               3.0|                  3|
|   -34.0|    -66.0|2025|                 1|                         3|               3.0|                  3|
|   -34.0|   -65.75|2025|                 1|                         3|               3.0|                  3|
|   -34.0|    -65.5|2025|                 1|                         3|               3.0|                  3|
|